# Publishing and Sharing

Building a dashboard on your local machine is only the development phase. A visualization yields zero business value until it is securely deployed, reliably updated, and easily accessible to decision-makers. 

The transition from a local file to an enterprise asset involves **Deployment Architecture**, **Data Governance**, and **Distribution Strategy**. 

In this lesson, we will explore the mechanisms used to publish analytical assets and simulate one of the most critical security features in enterprise reporting: **Row-Level Security (RLS)**.

Let's set up a Python sandbox to simulate a Business Intelligence (BI) server managing user access.

In [1]:
import pandas as pd

# 1. Simulate an Enterprise Sales Dataset
data = {
    'Transaction_ID': [1001, 1002, 1003, 1004, 1005, 1006],
    'Region': ['North', 'North', 'South', 'East', 'West', 'West'],
    'Client': ['Alpha Corp', 'Beta LLC', 'Gamma Inc', 'Delta Co', 'Epsilon Ltd', 'Zeta PLC'],
    'Revenue': [50000, 75000, 120000, 45000, 90000, 110000]
}
df_enterprise = pd.DataFrame(data)

print("✅ Enterprise Dataset Loaded on the Simulated Server.")
display(df_enterprise)

✅ Enterprise Dataset Loaded on the Simulated Server.


,Transaction_ID,Region,Client,Revenue
0,1001,North,Alpha Corp,50000
1,1002,North,Beta LLC,75000
2,1003,South,Gamma Inc,120000
3,1004,East,Delta Co,45000
4,1005,West,Epsilon Ltd,90000
5,1006,West,Zeta PLC,110000


# 1. Deployment Environments
When you create a dashboard in tools like Tableau or Power BI, you are typically working in a "Desktop" environment. To share this with the organization, it must be published to a centralized environment.

* **On-Premises Server**: The dashboard and its data are hosted on physical servers owned and maintained by your company. This is common in highly regulated industries (e.g., healthcare, finance) where strict data residency laws apply.
* **Cloud Environments**: The dashboard is hosted on a vendor's cloud infrastructure (e.g., Tableau Cloud, Power BI Service). This reduces internal IT maintenance and allows for seamless scaling.

# 2. Row-Level Security (RLS)
When publishing a global dashboard, you face a governance challenge: The Chief Revenue Officer needs to see the entire world, but the North Region Manager should *only* be permitted to see data for the North Region. 

Instead of building two separate dashboards, BI developers utilize **Row-Level Security (RLS)**. RLS intercepts the user's login credentials and dynamically filters the underlying data model before the dashboard renders.

Let's simulate RLS in Python.

In [2]:
# 2. Simulate a Server's Row-Level Security (RLS) Engine
def render_dashboard_with_rls(dataframe, username, role, assigned_region=None):
    """
    Simulates a BI Server validating user credentials and filtering data.
    """
    print(f"--- Authenticating User: {username} ({role}) ---")
    
    if role == 'Executive':
        # Executives have unrestricted access
        secure_data = dataframe.copy()
        print("🔓 Access Granted: Global View")
        
    elif role == 'Regional Manager':
        # Managers are restricted to their assigned region
        secure_data = dataframe[dataframe['Region'] == assigned_region].copy()
        print(f"🔒 Access Restricted: {assigned_region} Region Only")
        
    else:
        # Unauthorized roles receive no data
        secure_data = pd.DataFrame()
        print("🚫 Access Denied.")
        
    return secure_data

# Test Case A: The Chief Revenue Officer logs in
cro_view = render_dashboard_with_rls(df_enterprise, username="j.smith", role="Executive")
display(cro_view)

# Test Case B: The Western Regional Manager logs in
west_mgr_view = render_dashboard_with_rls(df_enterprise, username="a.davis", role="Regional Manager", assigned_region="West")
display(west_mgr_view)

--- Authenticating User: j.smith (Executive) ---
🔓 Access Granted: Global View


,Transaction_ID,Region,Client,Revenue
0,1001,North,Alpha Corp,50000
1,1002,North,Beta LLC,75000
2,1003,South,Gamma Inc,120000
3,1004,East,Delta Co,45000
4,1005,West,Epsilon Ltd,90000
5,1006,West,Zeta PLC,110000


--- Authenticating User: a.davis (Regional Manager) ---
🔒 Access Restricted: West Region Only


,Transaction_ID,Region,Client,Revenue
4,1005,West,Epsilon Ltd,90000
5,1006,West,Zeta PLC,110000


*(Notice how both users access the exact same Python function—or "dashboard"—but the output is fundamentally different based on their security profile. This prevents data leakage and ensures compliance with internal governance policies.)*

# 3. Distribution Strategy (Push vs. Pull)
Once a dashboard is secured and published, you must determine how stakeholders will interact with it.

* **The Pull Method (Self-Service BI)**: Users navigate to a web portal, locate the dashboard, and actively filter the data to find answers. This requires a high degree of data literacy and proactive engagement from the user.
* **The Push Method (Subscriptions & Alerts)**: Instead of relying on executives to log in, the BI server automatically generates a static snapshot (PDF or image) of the dashboard and emails it to them on a scheduled cadence (e.g., every Monday at 8:00 AM). Furthermore, **Data Alerts** can be configured to "push" a notification via email or Slack only when a specific threshold is breached (e.g., Revenue drops below a defined target).

# 4. Extract Refreshes and Pipeline Automation
A published dashboard is only as reliable as its data pipeline. 

If your dashboard connects to a data extract, you must configure an **Extract Refresh Schedule** upon publishing. This instructs the server to wake up during off-peak hours, connect to the underlying data warehouse (e.g., Snowflake, SQL Server), query the latest records, and overwrite the old extract. If this scheduled task fails, the dashboard will display stale data, leading to misinformed business decisions.

---

## Real-World Use Case or Analogy:
Think of Publishing and Sharing like operating a **Secure Corporate Library**:

* **Local Development**: You are an author writing a manuscript in your private office. No one else can read it, and it provides no value to the public.
* **Publishing**: You move the completed manuscript to the Corporate Library (the BI Server), where it is cataloged and made available on the shelves.
* **Row-Level Security (RLS)**: The Librarian checks the ID badge of anyone who requests the book. If an entry-level employee requests the Financial Forecast, the Librarian hands them a heavily redacted copy. If the CFO requests it, they receive the full, unredacted version.
* **Subscriptions (Push Method)**: Instead of forcing busy executives to walk to the library every morning, the Librarian makes a photocopy of the executive summary and drops it directly on their desk before they arrive at work.